In [ ]:
from functools import lru_cache

# Building: (name, construction time, earnings per operating unit)
BUILDINGS = [
    ("T", 5, 1500),   # Theatre
    ("P", 4, 1000),   # Pub
    ("C", 10, 2000),  # Commercial Park
]


def max_profit(n):
    """
    Find the maximum earnings for n construction time units.
    Buildings are constructed sequentially.
    A building earns money after its construction is complete.
    Returns the maximum earnings and all unique building combinations that achieve it.
    """
    if not isinstance(n, int) or isinstance(n, bool) or n < 0:
        raise ValueError("Time must be a non-negative integer.")

    @lru_cache(maxsize=None)
    def solve(elapsed):
        if elapsed >= n:
            return 0, frozenset([(0, 0, 0)])

        best_profit = 0
        best_combinations = frozenset([(0, 0, 0)])

        for i, (_, duration, rate) in enumerate(BUILDINGS):
            finish_time = elapsed + duration

            if finish_time > n:
                continue

            # Earnings from this building after construction finishes.
            profit = (n - finish_time) * rate

            # Solve for the remaining time
            future_profit, future_combs = solve(finish_time)
            total_profit = profit + future_profit

            # Update list of counts for each combination from the future subproblem
            new_combs = []
            for comb in future_combs:
                lst = list(comb)
                lst[i] += 1
                new_combs.append(tuple(lst))
            current_combs = frozenset(new_combs)

            if total_profit > best_profit:
                best_profit = total_profit
                best_combinations = current_combs
            elif total_profit == best_profit and total_profit > 0:
                best_combinations = best_combinations.union(current_combs)

        return best_profit, best_combinations

    profit, combinations = solve(0)

    # Convert unique combinations into a list of structured dictionaries
    combinations_list = [
        {"T": c[0], "P": c[1], "C": c[2]} for c in sorted(list(combinations))
    ]

    return {
        "time_units": n,
        "maximum_earnings": profit,
        "combinations": combinations_list
    }


def display_result(n):
    result = max_profit(n)
    print(f"Input time: {result['time_units']}")
    print(f"Maximum earnings: ${result['maximum_earnings']}")
    print("Valid Combinations:")
    for comb in result["combinations"]:
        print(f"  T: {comb['T']}, P: {comb['P']}, C: {comb['C']}")
    print("-" * 35)


# Assignment examples
for time_units in [7, 8, 13, 49]:
    display_result(time_units)

Input time: 7
Maximum earnings: $3000
Valid Combinations:
  T: 0, P: 1, C: 0
  T: 1, P: 0, C: 0
-----------------------------------
Input time: 8
Maximum earnings: $4500
Valid Combinations:
  T: 1, P: 0, C: 0
-----------------------------------
Input time: 13
Maximum earnings: $16500
Valid Combinations:
  T: 2, P: 0, C: 0
-----------------------------------
Input time: 49
Maximum earnings: $324000
Valid Combinations:
  T: 8, P: 2, C: 0
  T: 9, P: 0, C: 0
-----------------------------------


In [ ]:
# Validate the assignment's expected results
assert max_profit(7)["maximum_earnings"] == 3000
assert max_profit(8)["maximum_earnings"] == 4500
assert max_profit(13)["maximum_earnings"] == 16500

# Validate time unit = 49 scenario requested by the client
result_49 = max_profit(49)
assert result_49["maximum_earnings"] == 324000

expected_49_combinations = [
    {"T": 8, "P": 2, "C": 0},
    {"T": 9, "P": 0, "C": 0}
]
# Check that both expected combinations are present
for expected in expected_49_combinations:
    assert expected in result_49["combinations"], f"Missing expected combination: {expected}"

# Edge cases
assert max_profit(0)["maximum_earnings"] == 0
assert max_profit(3)["maximum_earnings"] == 0
assert max_profit(4)["maximum_earnings"] == 0
assert max_profit(5)["maximum_earnings"] == 1000 # Pub finishes at t=4, earns for 1 unit: 1 * 1000 = 1000
assert max_profit(6)["maximum_earnings"] == 2000 # Pub finishes at t=4, earns for 2 units: 2 * 1000 = 2000

print("All Max Profit tests (including n=49 requirements) passed!")

All Max Profit tests (including n=49 requirements) passed!
